In [18]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/cleaned_data.csv")

In [19]:
df["TransactionAmt_log"] = np.log1p(df["TransactionAmt"])
df['TransactionAmt_decimal'] = df['TransactionAmt'] % 1
df['TransactionAmt_isround'] = (df['TransactionAmt'] % 1 == 0).astype(int)

In [22]:
df['hour'] = df['TransactionDT'] % 86400 // 3600
df['day']  = df['TransactionDT'] // 86400 % 7
df['is_night'] = df['hour'].apply(lambda x: 1 if x < 6 or x > 22 else 0)

In [23]:
# How frequently does each card appear?
# High frequency = more data = better fraud detection
df['card1_freq'] = df['card1'].map(df['card1'].value_counts())
df['card2_freq'] = df['card2'].map(df['card2'].value_counts())

# Card + amount combination
df['card1_amt_mean'] = df.groupby('card1')['TransactionAmt'].transform('mean')
df['card1_amt_std']  = df.groupby('card1')['TransactionAmt'].transform('std')

# How different is this transaction from card's normal amount?
df['amt_vs_card1_mean'] = df['TransactionAmt'] / (df['card1_amt_mean'] + 1)